# Semaine 1 : Introduction aux Systèmes Temps Réel

## Objectifs pédagogiques
- Comprendre ce qu'est un système temps réel et ses contraintes.
- Distinguer les systèmes temps réel dur, mou et ferme.
- Identifier les caractéristiques de déterminisme et de sûreté de fonctionnement.
- Simuler une tâche périodique simple pour visualiser le respect des échéances.

## 1. Définition d'un système temps réel
Un **système temps réel** est un système dont la **correction** ne dépend pas seulement de la justesse des résultats calculés, mais aussi du **moment** où ces résultats sont produits. Le non-respect d'une contrainte temporelle (**échéance**) peut entraîner une défaillance du système.

**Exemples** :
- Contrôle de vol d'un avion (avionique)
- ABS dans une automobile
- Bras robotique industriel
- Système de régulation cardiaque (pacemaker)

## 2. Exemples de systèmes temps réel
### Avionique
- Gestion des commandes de vol, navigation, communication.
- Contraintes : latence très faible, haute fiabilité, certification stricte (DO-178C).

### Automobile
- ABS, contrôle moteur, airbags, assistance à la conduite.
- Contraintes : temps de réponse court (quelques ms), robustesse aux perturbations.

### Robotique
- Asservissement de position, vision, planification de trajectoire.
- Contraintes : périodicité précise, synchronisation entre capteurs et actionneurs.

## 3. Contraintes temporelles
- **Échéance (deadline)** : instant au plus tard où la tâche doit avoir terminé son exécution.
- **Période (period)** : intervalle de temps entre deux activations successives d'une tâche périodique.
- **Temps d'exécution au pire des cas (execution time) ou capacité** : durée nécessaire au processeur pour exécuter la tâche sans interruption.
- **Latence** (temps de réponse): délai entre l'événement déclencheur et la réponse du système.

Un système temps réel doit garantir que **toutes les échéances sont respectées**.

## 4. Classification des systèmes temps réel
| Type | Conséquence d'un dépassement d'échéance | Exemple |
|------|------------------------------------------|---------|
| **Temps réel dur** | Catastrophique (perte de vie, dégâts matériels) | Commande de vol, pacemaker |
| **Temps réel ferme** | Résultat inutile, mais pas de catastrophe | Streaming vidéo, réservation en ligne |
| **Temps réel mou** | Résultat dégradé, mais encore utilisable | Jeu vidéo, interface graphique |

## 5. Déterminisme et sûreté de fonctionnement
- **Déterminisme** : capacité du système à fournir une réponse dans un intervalle de temps borné et prévisible. Un système temps réel doit être déterministe temporellement.
- **Sûreté de fonctionnement** : ensemble de propriétés :
  - *Fiabilité* : probabilité de fonctionner sans défaillance sur une durée donnée.
  - *Disponibilité* : aptitude à être en état de fonctionnement.
  - *Sécurité* : absence de conséquences catastrophiques.
  - *Maintenabilité* : facilité de réparation et de mise à jour.

---
## Activité 1 : Classifier des systèmes
Pour chacun des systèmes suivants, indiquez s'il s'agit d'un système temps réel **dur**, **ferme** ou **mou** et justifiez brièvement.

1. Système de freinage ABS d'une voiture.
2. Lecteur vidéo en streaming (ex. Netflix).
3. Système de navigation GPS dans un smartphone.
4. Contrôleur de température d'un four industriel.
5. Jeu vidéo multijoueur en ligne.

**Répondez dans la cellule Markdown ci-dessous.**

*Votre réponse ici...*

---
## Activité 2 : Simulation d'une tâche périodique

Une tâche périodique est caractérisée par :
- **C** : temps d'exécution au pire des cas (constant) ou capacité de la tâche
- **T** : période (intervalle entre deux activations)
- **D** : échéance relative (souvent D = T) appelé encore delai critique
- **d** : échéance absolue (d = r + D), r = instant d'activation de la tache périodique

La simulation ci-dessous modélise l'exécution d'une tâche sur un processeur unique. La tâche est relâchée (réveillée) à chaque période et doit terminer avant son échéance. Le processeur exécute la tâche de manière continue dès qu'elle est prête.

Modifiez les valeurs de `C` et `T` et observez si la tâche respecte toutes ses échéances.

In [ ]:
def simuler_tache_periodique(C, T, duree_simulation):
    """
    Simule une tâche périodique de temps d'exécution C et période T.
    Retourne le statut de l'échéance de chaque instance libérée.
    """
    if C <= 0 or T <= 0 or duree_simulation <= 0:
        raise ValueError("C, T et duree_simulation doivent être strictement positifs")

    temps = 0
    prochaine_activation = 0
    instances = []
    resultats = []

    while temps < duree_simulation:
        # Chaque activation crée une nouvelle instance, conservée dans la file.
        if temps >= prochaine_activation:
            instances.append({
                "echeance": prochaine_activation + T,
                "reste": C,
            })
            prochaine_activation += T

        # Une seule instance s'exécute à la fois, dans l'ordre d'activation.
        if instances:
            instance = instances[0]
            instance["reste"] -= 1
            if instance["reste"] == 0:
                fin_execution = temps + 1
                statut = "Respectée" if fin_execution <= instance["echeance"] else "Violée"
                resultats.append(statut)
                instances.pop(0)

        temps += 1

    # Toute instance encore en attente a nécessairement dépassé son échéance
    # ou ne peut pas être évaluée dans la durée de simulation.
    resultats.extend("Violée" for _ in instances)
    return resultats

# Paramètres à modifier
C = 5   # temps d'exécution
T = 5   # période
duree = 20

res = simuler_tache_periodique(C, T, duree)
print(f"Résultats pour C={C}, T={T} :")
for i, r in enumerate(res, 1):
    print(f"  Instance {i}: échéance {r}")

In [ ]:
# Vérification du cas C > T et de l'échéance atteinte exactement.
assert simuler_tache_periodique(6, 5, 20) == ["Violée"] * 4
assert simuler_tache_periodique(5, 5, 15) == ["Respectée"] * 3
print("Tests C > T et C = T : OK")

---
## 6. Représentation temporelle avec Matplotlib

Le graphique ci-dessous représente l'exécution de la tâche pour `C = 3` et `T = 5`. Chaque activation démarre aux instants 0, 5, 10 et 15 ; les trois unités de temps d'exécution sont suivies d'une échéance respectée.

Modifiez les valeurs de `C` et `T` dans la cellule Python pour observer l'impact sur l'ordonnancement.

In [ ]:
import matplotlib.pyplot as plt

# Paramètres de la tâche
C = 3   # temps d'exécution
T = 5   # période
D = T   # échéance relative
duree = 20

activations = list(range(0, duree, T))
hauteur_execution = 0.2
y_base = 0
y_execution = y_base + hauteur_execution / 2
y_fleche_haut = 0.35

fig, ax = plt.subplots(figsize=(11, 3.4))

for numero, activation in enumerate(activations, 1):
    fin_execution = min(activation + C, duree)
    ax.barh(
        y=y_execution,
        width=fin_execution - activation,
        left=activation,
        height=hauteur_execution,
        color="#2f80ed",
        edgecolor="#1c4f91",
        label="Exécution" if numero == 1 else "",
    )
    ax.text(
        activation + (fin_execution - activation) / 2,
        y_execution,
        f"Instance {numero}",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
    )

    echeance = activation + D
    style_fleche = "<->" if D == T else "-|>"
    ax.annotate(
        "",
        xy=(echeance, y_base + 0.02),
        xytext=(echeance, y_fleche_haut),
        arrowprops={
            "arrowstyle": style_fleche,
            "color": "#d64545",
            "linestyle": "--",
            "linewidth": 1.2,
        },
    )


ax.annotate(
    "",
    xy=(T, y_periode),
    xytext=(0, y_periode),
    arrowprops={"arrowstyle": "<->", "color": "#159957", "linewidth": 1.5},
)
ax.text(
    T / 2,
    y_periode + 0.08,
    f"Période T = {T}",
    ha="center",
    va="bottom",
    color="#159957",
    fontweight="bold",
 )

# Entrées fictives pour conserver les échéances dans la légende
ax.plot([], [], color="#d64545", linestyle="--", marker=">", label="Échéance")

# L'axe des abscisses coïncide avec le bas des rectangles
ax.spines["bottom"].set_position(("data", y_base))
ax.spines["top"].set_visible(False)
ax.set_xlim(0, duree)
ax.set_ylim(-0.1, 0.98)
ax.set_yticks([y_execution])
ax.set_yticklabels(["Processeur"])
ax.set_xticks(range(0, duree + 1, T))
ax.set_xlabel("Temps")
ax.set_title(f"Ordonnancement d'une tâche périodique (C={C}, T={T}, D={D})")
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

---
## Activité 3 : Simulation de deux tâches périodiques

On considère deux tâches périodiques exécutées sur un processeur unique :

- **Tâche A** : temps d'exécution `C₁` et période `T₁` ;
- **Tâche B** : temps d'exécution `C₂` et période `T₂`.

À chaque instant, le processeur choisit parmi les tâches prêtes celle dont la période est la plus courte (ordonnancement Rate Monotonic). La simulation affiche l'utilisation du processeur et indique si les échéances sont respectées.

Modifiez les valeurs de `C₁`, `T₁`, `C₂` et `T₂`, puis observez l'effet sur l'ordonnancement.

In [ ]:
import matplotlib.pyplot as plt

# Paramètres modifiables des deux tâches
C1, T1 = 2, 5
C2, T2 = 1, 8
duree_simulation = 40

taches = {
    "Tâche A": {"C": C1, "T": T1, "couleur": "#2f80ed"},
    "Tâche B": {"C": C2, "T": T2, "couleur": "#f2994a"},
}

utilisation = sum(parametres["C"] / parametres["T"] for parametres in taches.values())
print(f"Utilisation processeur : {utilisation:.1%}")

instances = []
ordonnancement = []

for temps in range(duree_simulation):
    # Relâchement des nouvelles instances
    for nom, parametres in taches.items():
        if temps % parametres["T"] == 0:
            instances.append({
                "nom": nom,
                "T": parametres["T"],
                "activation": temps,
                "echeance": temps + parametres["T"],
                "reste": parametres["C"],
                "fin": None,
            })

    instances_pretes = [
        instance
        for instance in instances
        if instance["activation"] <= temps and instance["reste"] > 0
    ]

    if instances_pretes:
        # Rate Monotonic : la période la plus courte est prioritaire
        instance_active = min(instances_pretes, key=lambda instance: instance["T"])
        instance_active["reste"] -= 1
        ordonnancement.append(instance_active["nom"])
        if instance_active["reste"] == 0:
            instance_active["fin"] = temps + 1
    else:
        ordonnancement.append("Inactif")

# Bilan des échéances
for instance in instances:
    instance["respectee"] = (
        instance["fin"] is not None
        and instance["fin"] <= instance["echeance"]
    )
    etat = "respectée" if instance["respectee"] else "violée"
    print(
        f"{instance['nom']} activée à t={instance['activation']} : "
        f"échéance {etat}"
    )

# Construction des segments contigus du diagramme temporel
segments = []
debut_segment = 0
nom_segment = ordonnancement[0]

for instant in range(1, duree_simulation + 1):
    changement = instant == duree_simulation or ordonnancement[instant] != nom_segment
    if changement:
        segments.append((nom_segment, debut_segment, instant - debut_segment))
        if instant < duree_simulation:
            debut_segment = instant
            nom_segment = ordonnancement[instant]

fig, ax = plt.subplots(figsize=(12, 3.2))

for nom, debut, largeur in segments:
    couleur = "#d9d9d9" if nom == "Inactif" else taches[nom]["couleur"]
    ax.barh(
        y=0.25,
        width=largeur,
        left=debut,
        height=0.2,
        color=couleur,
        edgecolor="white",
    )
    if nom != "Inactif":
        ax.text(
            debut + largeur / 2,
            0.25,
            nom,
            ha="center",
            va="center",
            color="white",
            fontsize=8,
        )

for nom, parametres in taches.items():
    for echeance in range(parametres["T"], duree_simulation + 1, parametres["T"]):
        ax.axvline(
            echeance,
            color=parametres["couleur"],
            linestyle=":",
            linewidth=0.9,
            alpha=0.7,
        )

ax.spines["bottom"].set_position(("data", 0))
ax.spines["top"].set_visible(False)
ax.set_xlim(0, duree_simulation)
ax.set_ylim(-0.1, 0.8)
ax.set_yticks([0.25])
ax.set_yticklabels(["Processeur"])
ax.set_xticks(range(0, duree_simulation + 1, 5))
ax.set_xlabel("Temps")
ax.set_title("Ordonnancement de deux tâches périodiques")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Questions :**
1. Que se passe-t-il si C > T ?
2. Que se passe-t-il si C = T ?
3. Que se passe-t-il si C < T ?
4. À partir de quelle valeur de C (pour une période T donnée) l'ordonnancement devient-il impossible ?

---
## Exercice 1 : Questions de compréhension

1. Donnez une définition d'un système temps réel.
2. Expliquez la différence entre temps réel dur et temps réel mou avec un exemple pour chacun.
3. Qu'est-ce que le déterminisme temporel et pourquoi est-il important dans un système temps réel ?
4. Citez au moins trois domaines d'application des systèmes temps réel.
5. Quels sont les paramètres temporels principaux d'une tâche périodique ? Définissez-les.

---
## Exercice 2 : Analyse d'un ordonnancement simple (après week4)

Considérez une tâche périodique avec les caractéristiques suivantes :
- Temps d'exécution C = 2 ms
- Période T = 10 ms
- Échéance D = 10 ms

1. Calculez le taux d'utilisation processeur de cette tâche.
2. Combien d'instances de cette tâche peuvent être exécutées en une seconde ?
3. Si le processeur exécute également une autre tâche périodique de C = 4 ms et T = 20 ms, le système est-il ordonnançable avec l'algorithme Rate Monotonic (RM) ? Justifiez en utilisant la condition suffisante de Liu & Layland.

**Répondez dans la cellule Markdown ci-dessous.**

*Votre réponse ici...*

---
## TP : Prise en main d'un RTOS (FreeRTOS)

### Objectif
Familiarisez-vous avec l'environnement de développement d'un système d'exploitation temps réel (RTOS) en créant deux tâches simples qui s'exécutent de manière périodique.

### Matériel / Logiciel
- Carte de développement (ex. STM32 Nucleo, ESP32) ou simulateur (ex. QEMU, Wokwi).
- FreeRTOS installé (ou tout autre RTOS : Zephyr, RTX, etc.).
- Environnement de compilation (gcc, PlatformIO, STM32CubeIDE, etc.).

### Étapes
1. Créez un projet avec FreeRTOS.
2. Définissez deux tâches :
   - **Tâche 1** : clignote une LED toutes les 500 ms.
   - **Tâche 2** : incrémente un compteur et l'affiche sur la console toutes les 1 s.
3. Configurez les priorités des tâches (par exemple, tâche 1 priorité haute, tâche 2 priorité basse).
4. Lancez le programme et vérifiez que les deux tâches s'exécutent correctement.
5. Modifiez les périodes et les priorités, observez l'impact sur l'exécution.

### Questions
- Comment le RTOS gère-t-il l'alternance entre les deux tâches ?
- Que se passe-t-il si les deux tâches ont la même priorité ?
- Comment pouvez-vous mesurer le temps d'exécution de chaque tâche ?

## Références
- Buttazzo, G. – *Hard Real-Time Computing Systems*, Chapitre 1-2.
- Liu, J.W.S. – *Real-Time Systems*, Chapitre 1.